# 01. 컨텍스트와 토큰 예산

모델 호출에는 입력과 출력이 함께 들어갈 공간이 필요합니다. 이 노트북에서는 입력 길이를 호출 전에 추정하고, 실제 API 사용량과 비교합니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/01-컨텍스트와-토큰-예산.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

아래 셀은 현재 작업 폴더에서 상위 폴더로 올라가며 `.env`를 찾습니다. 키 값은 화면에 출력하지 않습니다.

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
import os

from dotenv import find_dotenv, load_dotenv

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 출력 공간을 먼저 예약합니다

전체 예산에서 출력 예약분과 안전 여유를 빼면 입력에 사용할 수 있는 범위가 나옵니다.

In [4]:
# total은 전체 한도, output_reserve는 답변용 예약분입니다.
# safety_margin은 토큰 추정 오차를 위한 여유입니다.
def input_budget(total: int, output_reserve: int, safety_margin: int) -> int:
    available = total - output_reserve - safety_margin

    if available < 0:
        raise ValueError("출력 예약분과 여유가 전체 예산보다 큽니다.")

    return available


### 세 값을 넣어 반환값을 확인합니다

전체 한도 `4096`에서 출력 예약분 `300`과 안전 여유 `128`을 뺍니다. 반환된 숫자는 `available`에 저장됩니다.

In [5]:
# 세 인자는 함수 정의에 적힌 순서대로 전달합니다.
available = input_budget(4096, 300, 128)
print("입력에 사용할 수 있는 토큰:", available)

입력에 사용할 수 있는 토큰: 3668


## 3. 로컬 추정값을 확인합니다

토크나이저는 문장을 모델이 세는 작은 단위인 토큰으로 나누어, 호출 전에 입력 크기를 추정합니다. 실제 호출 뒤에는 응답 객체의 `AIMessage.usage_metadata`에서 제공자가 센 토큰 수를 확인합니다.

In [6]:
# question은 질문이고 evidence는 답변 근거입니다.
import tiktoken

question = "미르시립도서관의 정기 휴관일은 언제인가요?"
evidence = "[E01 p.2] 정기 휴관일은 매주 월요일입니다."
prompt = f"""근거:
{evidence}

질문: {question}"""
print(prompt)

근거:
[E01 p.2] 정기 휴관일은 매주 월요일입니다.

질문: 미르시립도서관의 정기 휴관일은 언제인가요?


### 같은 문자열을 토큰으로 바꿉니다

`encode()`는 문자열을 토큰 ID 목록으로 반환합니다. 목록 길이를 세어 호출 전 입력 크기를 추정합니다.

In [7]:
# 실제 API 입력에는 system 메시지와 채팅 형식용 토큰도 더해집니다.
encoding = tiktoken.get_encoding("o200k_base")
token_ids = encoding.encode(prompt)
estimated = len(token_ids)
print("로컬 추정 입력 토큰:", estimated)
print("예산 안에 있음:", estimated <= available)

로컬 추정 입력 토큰: 42
예산 안에 있음: True


## 4. 실제 API 사용량을 비교합니다

`AIMessage.text`와 `usage_metadata`를 함께 확인합니다. 출력은 실행마다 달라질 수 있습니다.

### 모델에 전달할 메시지를 만듭니다

`system`에는 답변 규칙을, `user`에는 앞에서 만든 질문을 넣습니다. 셀을 실행한 뒤 역할 순서를 확인합니다.

In [8]:
# messages는 역할과 내용을 짝지은 목록입니다.
# system 지시를 user 질문보다 먼저 둡니다.
messages = [
    ("system", "근거에 있는 정보만 사용하고 출처를 표시하세요."),
    ("user", prompt),
]
roles = []
for role, content in messages:
    roles.append(role)
print("메시지 역할:", roles)

메시지 역할: ['system', 'user']


### 모델을 한 번 호출합니다

`invoke()`의 반환값은 `AIMessage`입니다. 사용량 정보가 없으면 이후 비교를 할 수 없으므로 즉시 오류를 냅니다.

In [9]:
# response는 모델의 답과 메타데이터를 가진 AIMessage입니다.
response = model.invoke(messages)
usage = response.usage_metadata

if usage is None:
    raise RuntimeError("모델 사용량 메타데이터가 없습니다.")

### 답과 실제 토큰 수를 관찰합니다

먼저 답변 원문을 읽고, 다음 두 줄에서 API가 보고한 입력·출력 토큰 수를 확인합니다. 답변 문구와 수치는 실행마다 달라질 수 있습니다.

In [10]:
# text는 답변 문자열이고 usage는 실제 호출의 토큰 사용량입니다.
print(response.text)
print("input_tokens:", usage.get("input_tokens"))
print("output_tokens:", usage.get("output_tokens"))

미르시립도서관의 정기 휴관일은 **매주 월요일**입니다. [E01 p.2]
input_tokens: 66
output_tokens: 32


## 5. 출력 예약분 한 값만 바꿔 비교합니다

전체 한도와 안전 여유는 유지하고 출력 예약분만 `300`에서 `600`으로 늘립니다. 입력 예산이 얼마나 줄어드는지 확인합니다.

In [11]:
# 두 호출은 output_reserve 값 하나만 다릅니다.
base_budget = input_budget(4096, 300, 128)
changed_budget = input_budget(4096, 600, 128)
print("기본 입력 예산:", base_budget)
print("변경 입력 예산:", changed_budget)
print("차이:", changed_budget - base_budget)

기본 입력 예산: 3668
변경 입력 예산: 3368
차이: -300


## 정리

출력 공간을 먼저 예약하고, 입력을 로컬에서 추정한 뒤, 실제 사용량은 API 응답에서 확인합니다.

## 실패 재현: 출력 공간을 남기지 않습니다

전체 예산보다 출력 예약분과 여유를 크게 설정하면 준비한 입력을 넣을 공간이 없습니다. 같은 입력 길이를 수정 전후 허용량과 비교합니다.

In [12]:
# planned_input은 모델에 넣으려는 입력의 길이입니다.
# 잘못된 전체 예산에서는 입력 허용량을 계산할 수 없습니다.
planned_input = 500
bad_total = 300
try:
    bad_available = input_budget(bad_total, 300, 128)
except ValueError as error:
    print("관찰한 실패:", error)
    print("준비한 입력 토큰:", planned_input, "> 허용량 없음")
else:
    raise AssertionError("예산 오류가 발생해야 합니다.")

관찰한 실패: 출력 예약분과 여유가 전체 예산보다 큽니다.
준비한 입력 토큰: 500 > 허용량 없음


## 수정: 전체 예산을 복원합니다

같은 500토큰 입력은 유지합니다. 모델 한도를 바꾸는 것이 아니라 수업에서 정한 작업 예산을 4,096토큰으로 고친 뒤 입력 허용량 안에 들어가는지 확인합니다.

In [13]:
# 입력 길이는 그대로 두고 전체 예산만 복원합니다.
# 출력 예약분과 여유를 빼도 500토큰 입력을 넣을 수 있어야 합니다.
fixed_total = 4096
fixed_available = input_budget(fixed_total, 300, 128)
assert fixed_available == 3668
assert planned_input <= fixed_available
print("준비한 입력 토큰:", planned_input)
print("수정 후 입력 허용량:", fixed_available)
print("입력 가능:", planned_input <= fixed_available)

준비한 입력 토큰: 500
수정 후 입력 허용량: 3668
입력 가능: True
